# TMG301 · Nhóm 3 — Multilingual Transformer Việt/Pháp → Anh

Chạy toàn bộ Phần 3 + kết quả Phần 4 trên **1 GPU T4 miễn phí** (Kaggle hoặc Colab).

**Cách dùng**
1. Lần đầu để `QUICK_TEST = True` (ô *Cấu hình*) → chạy tất cả (~30–45 phút) để chắc chắn không lỗi.
2. Đổi `QUICK_TEST = False` → chạy tất cả lần nữa. Kaggle: **Save Version → Save & Run All (Commit)**, chạy nền, đóng trình duyệt được. Colab: **Runtime → Run all**.
3. Bị ngắt giữa chừng → chạy tất cả lại: bước nào xong rồi sẽ tự bỏ qua.
4. Lỗi ở đâu notebook **dừng ngay ở đó** và in log; gửi ~30 dòng cuối cho Claude.

In [ ]:
!nvidia-smi

## 0. Lấy mã nguồn
- **Kaggle:** *Add Input → Upload* file `mt_multilingual.zip` (Kaggle có thể tự giải nén, notebook xử lý cả hai trường hợp).
- **Colab:** đặt `mt_multilingual.zip` trong Google Drive, thư mục `tmg301_mt`.

In [ ]:
import os, glob, shutil, time, subprocess
ON_COLAB = os.path.exists('/content') and not os.path.exists('/kaggle')
ON_KAGGLE = os.path.exists('/kaggle/working')
BATCH = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch'   # Kaggle "Save & Run All": chạy nền, không ai bấm
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR, ROOT = '/content/drive/MyDrive/tmg301_mt/work', '/content'
elif ON_KAGGLE:
    WORKDIR, ROOT = '/kaggle/working/work', '/kaggle/working'
else:
    WORKDIR, ROOT = os.path.abspath('work'), os.getcwd()

CODE = f'{ROOT}/mt_multilingual'
zips = (glob.glob(f'{ROOT}/mt_multilingual.zip') + glob.glob('/content/drive/MyDrive/tmg301_mt/mt_multilingual.zip')
        + glob.glob('/kaggle/input/**/mt_multilingual.zip', recursive=True))
unzipped = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/mtm/__init__.py', recursive=True)]
if zips:
    shutil.rmtree(CODE, ignore_errors=True)
    subprocess.run(['unzip', '-oq', zips[0], '-d', ROOT], check=True)
elif unzipped:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(unzipped[0], CODE)
else:
    raise FileNotFoundError('Không thấy mt_multilingual.zip — Colab: Drive/tmg301_mt/ · Kaggle: Add Input → Upload')
os.chdir(CODE)
print('code:', os.getcwd(), '| workdir gốc:', WORKDIR, '| chạy nền:' , BATCH)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q underthesea || echo "Không cài được underthesea → độ hiếm R tách theo âm tiết"
!pip uninstall -y -q torchao   # Kaggle cài sẵn torchao 0.10 quá cũ → peft báo lỗi; dự án không cần torchao


## Cấu hình
- `QUICK_TEST = True`: dữ liệu nhỏ, vài trăm bước — chỉ để bắt lỗi, **không dùng số liệu cho slide**.
- `PLAN = 'nhanh'` (~4–5 giờ T4, khuyên dùng) hoặc `'day_du'` (~6–7 giờ, nhiều bước hơn).

In [ ]:
import yaml
QUICK_TEST = True        # chạy thử xong → False
PLAN = 'nhanh'           # 'nhanh' | 'day_du'

cfg = yaml.safe_load(open('configs/default.yaml', encoding='utf-8'))
cfg['workdir'] = WORKDIR
# Chưa được duyệt quyền PhoMT? bỏ dấu # ở dòng dưới để dùng dữ liệu dự phòng (OPUS TED2020 Việt–Anh)
# cfg['data']['vi_source'] = 'opus_ted2020'
if PLAN == 'nhanh':
    cfg['train'].update(stage1_steps=4000, stage2_steps=800, eval_every=400, eval_size=200, patience=3)
    cfg['scratch'].update(steps=6000)
    cfg['data'].update(fr_test_size=500, retrieval_eval_size=500)
    cfg['decode'].update(num_beams=4)
if QUICK_TEST:
    cfg['workdir'] = WORKDIR + '_test'
    cfg['data'].update(vi_train_size=2000, vi_dev_size=200, vi_test_size=200, fr_train_size=5000,
                       fr_dev_size=100, fr_test_size=200, retrieval_eval_size=200)
    cfg['train'].update(stage1_steps=200, stage2_steps=50, eval_every=100, eval_size=50)
    cfg['scratch'].update(steps=300, warmup=50)
W = cfg['workdir']
yaml.safe_dump(cfg, open('configs/run.yaml', 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print(('CHẠY THỬ NHANH' if QUICK_TEST else 'CHẠY THẬT'), '| gói:', PLAN, '| workdir:', W)

T0 = time.time()
def run(cmd):
    # Chạy một bước; lỗi thì DỪNG notebook ngay (lệnh ! của notebook không tự dừng khi lỗi)
    print(f'\n$ {cmd}', flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, PYTHONUNBUFFERED='1', HF_HUB_DISABLE_PROGRESS_BARS='1',
                                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True'))
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'LỖI ở bước: {cmd}  →  gửi ~30 dòng log phía trên cho Claude')
    print(f'✓ xong sau {(time.time() - t) / 60:.1f} phút (tổng {(time.time() - T0) / 60:.0f} phút)')

def done(*parts):
    return os.path.exists(os.path.join(W, *parts))

run('python tests/test_core.py')

In [ ]:
# Token Hugging Face (để tải PhoMT). Kaggle: Add-ons → Secrets → HF_TOKEN. Colab: biểu tượng 🔑 → HF_TOKEN, hoặc dán khi được hỏi.
token = os.environ.get('HF_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    token = token or UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
try:
    from google.colab import userdata
    token = token or userdata.get('HF_TOKEN')
except Exception:
    pass
if not token and not done('data', 'vi_en_train.jsonl') and cfg['data']['vi_source'] == 'phomt':
    if BATCH:
        raise RuntimeError('Thiếu HF_TOKEN: Add-ons → Secrets → thêm HF_TOKEN và bật cho notebook này')
    from getpass import getpass
    token = getpass('Dán token Hugging Face (hf_...): ').strip()
if token:
    os.environ['HF_TOKEN'] = token
    print('Đã có token Hugging Face')

## 1. Dữ liệu (slide 3.1) · 2. Kho Pháp–Anh (slide 3.6)

In [ ]:
if not done('data', 'vi_en_train.jsonl'):
    run('python -m mtm.data --config configs/run.yaml')
else:
    print('Dữ liệu đã có → bỏ qua')
if not done('memory', 'index.faiss'):
    run('python -m mtm.memory --config configs/run.yaml')
else:
    print('Kho Pháp–Anh đã có → bỏ qua')

## 3. Huấn luyện (slide 3.3, 3.4, 3.7)
A0 = Transformer nhỏ học từ đầu · A = mBART-50 + LoRA chỉ Việt–Anh · B = + Pháp–Anh học chung · C = B + mẫu kèm bằng chứng.
Mô hình nào đã huấn luyện xong (có `history.json`) sẽ được bỏ qua.

In [ ]:
for exp in ['A0', 'A', 'B', 'C']:
    if done('checkpoints', exp, 'history.json'):
        print(f'{exp} đã huấn luyện xong → bỏ qua')
        continue
    run(f'python -m mtm.train --config configs/run.yaml --exp {exp}')

In [ ]:
# (Tùy chọn, cho slide 3.4) quét nhiệt độ T = 1 và T = 2 — thêm ~2 giờ GPU
# for T in [1, 2]:
#     if not done('checkpoints', f'B_T{T}', 'history.json'):
#         run(f'python -m mtm.train --config configs/run.yaml --exp B --temperature {T}')

## 4. Chọn ngưỡng τ, ρ, δ trên dev (slide 3.5) · 5. Đánh giá trên test (Phần 4)

In [ ]:
if not done('checkpoints', 'C', 'thresholds.json'):
    run('python -m mtm.evaluate --config configs/run.yaml --calibrate --exp C')
for exp in ['A0', 'A', 'B', 'C']:
    if not done('results', f'{exp}_test.json'):
        run(f'python -m mtm.evaluate --config configs/run.yaml --exp {exp}')
if not done('results', 'retrieval.json'):
    run('python -m mtm.evaluate --config configs/run.yaml --retrieval')
run('python -m mtm.evaluate --config configs/run.yaml --summary')

In [ ]:
# (Tùy chọn) COMET — cài thêm thư viện, chạy lại đánh giá (~1 giờ). Nếu báo lỗi phiên bản: Restart rồi chạy lại ô này.
# run('pip install -q unbabel-comet')
# for exp in ['A0', 'A', 'B', 'C']:
#     run(f'python -m mtm.evaluate --config configs/run.yaml --exp {exp} --comet')
# run('python -m mtm.evaluate --config configs/run.yaml --summary')

## 6. Xuất mô hình cho app demo
Tạo `nhom3_mt.zip` (~1,6 GB): mô hình C đã gộp LoRA, kho Pháp–Anh, ngưỡng, kết quả, ví dụ câu khó.
Kaggle: tải ở tab **Output** của phiên bản. Colab: Google Drive → `tmg301_mt/work/export/`.

In [ ]:
run('python -m mtm.export --config configs/run.yaml --exp C')
z = os.path.join(W, 'export', 'nhom3_mt.zip')
print(f'File để tải về laptop: {z}  ({os.path.getsize(z) / 1e9:.2f} GB)')
print(f'Tổng thời gian: {(time.time() - T0) / 3600:.1f} giờ')

## 7. Demo nhanh bằng Gradio (tùy chọn — demo chính: python app.py ở thư mục gốc repo)

In [ ]:
if BATCH:
    print('Đang chạy nền → bỏ qua demo')
else:
    print('Bỏ dấu # ở dòng dưới để mở demo (dừng bằng nút ■)')
    # run('python app.py --config configs/run.yaml --exp C --share')